In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder,MinMaxScaler
from sklearn.metrics import f1_score, accuracy_score

In [183]:
# Load Data 
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

test_df.head()


,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade
0,593994,28781.05,0.049,626,11461.42,14.73,Female,Single,High School,Employed,Other,D5
1,593995,46626.39,0.093,732,15492.25,12.85,Female,Married,Master's,Employed,Other,C1
2,593996,54954.89,0.367,611,3796.41,13.29,Male,Single,Bachelor's,Employed,Debt consolidation,D1
3,593997,25644.63,0.110,671,6574.30,9.57,Female,Single,Bachelor's,Employed,Debt consolidation,C3
4,593998,25169.64,0.081,688,17696.89,12.80,Female,Married,PhD,Employed,Business,C1


In [184]:
# now that we have the data imported we should start by first getting all the unique types of vals in non int cols
gender_col            = train_df['gender']
marital_status_col    = train_df['marital_status']
educational_level_col = train_df['education_level']
employment_status_col = train_df['employment_status']
loan_purposes_col     = train_df['loan_purpose']
grade_subgrade_col    = train_df['grade_subgrade']

print(
    'Here are the following unique values in string cols\n',
    f'Gender Col: {gender_col.unique()}\n',
    f'Marital Status Col: {marital_status_col.unique()}\n',
    f'Educational Level Col: {educational_level_col.unique()}\n',
    f'Employment Status Col: {employment_status_col.unique()}\n',
    f'Loan Purposes Col: {loan_purposes_col.unique()}\n',
    f'Grade subgrade Col: {grade_subgrade_col.unique()}'
)

Here are the following unique values in string cols
 Gender Col: ['Female' 'Male' 'Other']
 Marital Status Col: ['Single' 'Married' 'Divorced' 'Widowed']
 Educational Level Col: ['High School' "Master's" "Bachelor's" 'PhD' 'Other']
 Employment Status Col: ['Self-employed' 'Employed' 'Unemployed' 'Retired' 'Student']
 Loan Purposes Col: ['Other' 'Debt consolidation' 'Home' 'Education' 'Vacation' 'Car'
 'Medical' 'Business']
 Grade subgrade Col: ['C3' 'D3' 'C5' 'F1' 'D1' 'D5' 'C2' 'C1' 'F5' 'D4' 'C4' 'D2' 'E5' 'B1'
 'B2' 'F4' 'A4' 'E1' 'F2' 'B4' 'E4' 'B3' 'E3' 'B5' 'E2' 'F3' 'A5' 'A3'
 'A1' 'A2']


In [185]:
# now we prepare our custom ranking for ordinal cols
grade_subgrade_ordered = [
    'F5', 'F4', 'F3', 'F2', 'F1',  
    'E5', 'E4', 'E3', 'E2', 'E1',  
    'D5', 'D4', 'D3', 'D2', 'D1',  
    'C5', 'C4', 'C3', 'C2', 'C1',  
    'B5', 'B4', 'B3', 'B2', 'B1',  
    'A5', 'A4', 'A3', 'A2', 'A1'   
]

educational_level_ordered = [
    'Other', 'High School', "Bachelor's", "Master's", 'PhD'
]

In [186]:
# here we intiliaze and fit the encoder
ordinal_categories = [
    grade_subgrade_ordered,
    educational_level_ordered
]

ordinal_encoder = OrdinalEncoder(
    categories=ordinal_categories
)

In [187]:
ordinal_cols = ['grade_subgrade', 'education_level']
ordinal_encoder.fit(train_df[ordinal_cols])
train_ordinal_encoded_array = ordinal_encoder.transform(train_df[ordinal_cols])

ordinal_encoder.fit(test_df[ordinal_cols])
test_ordinal_encoded_array = ordinal_encoder.transform(test_df[ordinal_cols])

train_ordinal_encoded_df = pd.DataFrame(
    train_ordinal_encoded_array, 
    columns=ordinal_cols, 
    index=train_df.index
)

test_ordinal_encoded_df = pd.DataFrame(
    test_ordinal_encoded_array, 
    columns=ordinal_cols, 
    index=test_df.index
)

train_df[ordinal_cols[0]] = train_ordinal_encoded_df[ordinal_cols[0]]
train_df[ordinal_cols[1]] = train_ordinal_encoded_df[ordinal_cols[1]]

test_df[ordinal_cols[0]] = test_ordinal_encoded_df[ordinal_cols[0]]
test_df[ordinal_cols[1]] = test_ordinal_encoded_df[ordinal_cols[1]]

test_df.head()


,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade
0,593994,28781.05,0.049,626,11461.42,14.73,Female,Single,1.0,Employed,Other,10.0
1,593995,46626.39,0.093,732,15492.25,12.85,Female,Married,3.0,Employed,Other,19.0
2,593996,54954.89,0.367,611,3796.41,13.29,Male,Single,2.0,Employed,Debt consolidation,14.0
3,593997,25644.63,0.110,671,6574.30,9.57,Female,Single,2.0,Employed,Debt consolidation,17.0
4,593998,25169.64,0.081,688,17696.89,12.80,Female,Married,4.0,Employed,Business,19.0


In [188]:
nominal_cols = ['marital_status', 'employment_status', 'loan_purpose', 'gender']
oneHot = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False) 
train_hot_encoded_array = oneHot.fit_transform(train_df[nominal_cols])
test_hot_encoded_array = oneHot.fit_transform(test_df[nominal_cols])

train_new_ohe_cols = oneHot.get_feature_names_out(nominal_cols)
test_new_ohe_cols = oneHot.get_feature_names_out(nominal_cols)

train_hot_encoded_df = pd.DataFrame(
    train_hot_encoded_array, 
    columns=train_new_ohe_cols, 
    index=train_df.index
)

test_hot_encoded_df = pd.DataFrame(
    test_hot_encoded_array, 
    columns=test_new_ohe_cols, 
    index=test_df.index
)

train_df = pd.concat([train_df.drop(columns=nominal_cols), train_hot_encoded_df], axis=1)
test_df = pd.concat([test_df.drop(columns=nominal_cols), test_hot_encoded_df], axis=1)


In [189]:
# init scalers 
minmaxScaler   = MinMaxScaler()
scalable_cols = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
scaled_arr = minmaxScaler.fit_transform(train_df[scalable_cols])
scaled_df = pd.DataFrame(
    scaled_arr,
    columns=scalable_cols,
    index=train_df.index
)
train_df = pd.concat([train_df.drop(columns=scalable_cols), scaled_df], axis=1)

test_scaled_arr = minmaxScaler.fit_transform(test_df[scalable_cols])
test_scaled_df = pd.DataFrame(
    test_scaled_arr,
    columns=scalable_cols,
    index=test_df.index
)
test_df = pd.concat([test_df.drop(columns=scalable_cols), test_scaled_df], axis=1)

In [190]:
drop_cols = ['id', 'loan_paid_back']
X_train = train_df.drop(drop_cols, axis=1)
y_train = train_df['loan_paid_back']

In [191]:
# init models 
models = {
    'DecisionTree': DecisionTreeClassifier(),
    'Ridge': RidgeClassifier(),
    'LogisticRegression': LogisticRegression(),
    'MLP': MLPClassifier(),
    'LGBM': lgb.LGBMClassifier(objective='binary'),
    'CatBoost': CatBoostClassifier(objective='Logloss'),
    'XGB': xgb.XGBClassifier(objective='binary:logistic')
}

In [192]:
model_param_config = {
    'DecisionTree': {
        'criterion': ['gini', 'entropy'],
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_leaf': [1, 2, 5, 10]
    },
    'Ridge': {
        'alpha': [0.1, 1.0, 10.0, 100.0]
    },
    'LogisticRegression': {
        'C': [0.1, 1, 10, 100],
        'penalty': ['l2'] 
    },
    'MLP': {
        'hidden_layer_sizes': [(50,), (100,), (50, 50)],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate_init': [0.001, 0.01]
    },
    'LGBM': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'num_leaves': [20, 31, 40], 
    },
    'CatBoost': {
        'iterations': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'depth': [4, 6, 8] 
    },
    'XGB': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
        'gamma': [0, 0.1, 0.5]
    }
}

In [193]:
# results = {}

# for model_name, model in models.items():
#     print(f" Starting RandomizedSearch for {model_name}")
    
#     params = model_param_config[model_name]
    
#     random_search = RandomizedSearchCV(
#         estimator=model,
#         param_distributions=params,
#         n_iter=25,
#         scoring='roc_auc',
#         cv=5,
#         random_state=42,
#         n_jobs=-1 
#     )
    
#     random_search.fit(X_train, y_train)

#     results[model_name] = {
#         'best_score': random_search.best_score_,
#         'best_params': random_search.best_params_,
#         'best_estimator': random_search.best_estimator_,
#         'error_score': random_search.error_score
#     }
    
#     print(f"Finished search for {model_name}. Best AUC: {results[model_name]['best_score']}")

# print("All searches complete.")

In [194]:
from sklearn.model_selection import train_test_split 
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.20)
lgbm_model = lgb.LGBMClassifier(objective='binary', num_leaves=40, n_estimators=300, learning_rate=0.1)
lgbm_model.fit(X_train, y_train)

results = lgbm_model.predict(X_test)



[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 379725, number of negative: 95470
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001251 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1341
[LightGBM] [Info] Number of data points in the train set: 475195, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.799093 -> initscore=1.380635
[LightGBM] [Info] Start training from score 1.380635


In [195]:
acc = accuracy_score(y_test, results)
f1  = f1_score(y_test, results)
acc

0.9058325406779518

In [196]:
f1

0.9431719471494536

In [205]:
# Make submission 
id = test_df['id']
x  = test_df.drop('id', axis=1)
submission_preds = lgbm_model.predict(x)
submission_df = pd.DataFrame({
    'id': id,
    'loan_paid_back': submission_preds
})

submission_df.to_csv('data/submission.csv', index=False)